In [1]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv("../openai.env")

True

In [2]:
persistent_directory="./db/chroma_db"

## 1. Recreating the Vector Store

In [3]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
db= Chroma(persist_directory=persistent_directory,
           embedding_function=embedding_model,
           collection_metadata={"hnsw:space":"cosine"})

## 2. Query Generation, Retrieval Engine

In [4]:
query = "How do plants turn sunlight into energy ?" 

#Return the top 3 matched documents
retriever= db.as_retriever(search_kwargs={"k":3})

# This is where a query turns into a vector by the embedding model
relevant_docs = retriever.invoke(query)

print(f"User Query: {query}")
print("---Context---")

for i, doc in enumerate(relevant_docs,1):
    print(f"Document{i}:\n{doc.page_content}\n")
    

User Query: How do plants turn sunlight into energy ?
---Context---
Document1:
Photosynthesis is a system of biological processes by which photopigment-bearing autotrophic organisms, such as most plants, algae and cyanobacteria, convert light energy—typically from sunlight—into the chemical energy necessary to fuel their metabolism. The term photosynthesis usually refers to oxygenic photosynthesis, a process that releases oxygen as a byproduct of water splitting. Photosynthetic organisms store the converted chemical energy within the bonds of intracellular organic compounds (complex compounds containing carbon), typically carbohydrates like sugars (mainly glucose, fructose and sucrose), starches, phytoglycogen and cellulose. When needing to use this stored energy, an organism's cells then metabolize the organic compounds through cellular respiration. Photosynthesis plays a critical role in producing and maintaining the oxygen content of the Earth's atmosphere, and it supplies most of t

## 3. Answer Generation

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [13]:
# We want to combine the query and the relevant document contents
combined_input = f"""Based on the following documents, please answer this question:{query}"

Documents:
{chr(10).join([f"-{doc.page_content}" for doc in relevant_docs])}

Please Provide a clear helpful answer, using only the information from these documents. If you can't find it say: Did not manage to find the document
"""


##  4. Create a ChatOpenAI model

In [14]:
model= ChatOpenAI(model="gpt-4o")

messages = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content=combined_input),]

    

## 5. Invoke the Model with the Combined Input

In [15]:
result = model.invoke(messages)

## 6. Display the full result and content only

In [16]:
print("\n --- Generated Response ---")

print("Content only:")
print(result.content)


 --- Generated Response ---
Content only:
Plants turn sunlight into energy through a process called photosynthesis. In this system, plants, which contain pigments like chlorophyll, capture light energy from the sun. This process primarily occurs in the chloroplasts, which are organelles within plant cells. 

1. **Light Absorption**: Plants absorb sunlight using chlorophyll, a pigment located within the chloroplasts, specifically embedded in complexes called antenna proteins in the thylakoids, which are part of the chloroplasts' internal structure. 

2. **Light-Dependent Reactions**: The absorbed light energy is used in light-dependent reactions to split water molecules, releasing oxygen gas. This process creates two important molecules: ATP (adenosine triphosphate) and NADPH (reduced nicotinamide adenine dinucleotide phosphate), which store energy.

3. **Light-Independent Reactions (Calvin Cycle)**: In the subsequent light-independent reactions, also known as the Calvin cycle, the ene